In [1]:
from run_stacking import *

2025-12-15 10:27:26.246239: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = 'Worms'

X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
print(X_train.shape, y_train.shape)

(181, 1, 900) (181,)


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import ExtraTreesClassifier
from aeon.transformations.collection.shapelet_based import (
    RandomDilatedShapeletTransform,
)
from aeon.transformations.collection.convolution_based import MultiRocket
import polars.selectors as cs
from time import perf_counter

from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from aeon.transformations.collection.feature_based import Catch22
from aeon.classification.interval_based import DrCIFClassifier
from aeon.classification.interval_based import RSTSF

class MultiScaler(BaseEstimator, TransformerMixin):
    """
    Applies different scalers to different feature groups.
    Features not in any group are ignored.
    
    Parameters
    ----------
    scalers : dict
        Dictionary mapping feature prefix to scaler instance.
        Example: {'hydra': SparseScaler(), 'multirocket': StandardScaler()}
    verbose : bool, default=False
        If True, print information about which features are scaled with which scaler.
    """
    def __init__(self, scalers, verbose=False):
        self.scalers = scalers
        self.verbose = verbose
    
    def fit(self, X, y=None):
        self.scalers_ = {}
        self.feature_groups_ = {}
        
        # Group columns by prefix
        for prefix, scaler in self.scalers.items():
            cols = [col for col in X.columns if col.startswith(prefix)]
            if cols:
                self.feature_groups_[prefix] = cols
                self.scalers_[prefix] = scaler
                self.scalers_[prefix].fit(X.select(cols).to_numpy())
                
                if self.verbose:
                    print(f"[MultiScaler] {len(cols)} '{prefix}' features -> {scaler.__class__.__name__}")
        
        # Store output column order
        self.output_cols_ = [col for prefix in self.scalers.keys() 
                            for col in self.feature_groups_.get(prefix, [])]
        
        if self.verbose:
            total_input = len(X.columns)
            total_output = len(self.output_cols_)
            ignored = total_input - total_output
            print(f"[MultiScaler] Total: {total_output}/{total_input} features kept, {ignored} ignored")
        
        return self
    
    def transform(self, X):
        parts = []
        for prefix in self.scalers_.keys():
            if prefix in self.feature_groups_:
                cols = self.feature_groups_[prefix]
                scaled = self.scalers_[prefix].transform(X.select(cols).to_numpy())
                parts.append(scaled)
        
        return np.hstack(parts) if parts else np.empty((len(X), 0))
    
    def get_feature_names_out(self, input_features=None):
        return np.array(self.output_cols_)

class NoScaler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X

class RidgeClassifierCVIndicator(RidgeClassifierCV):
    def predict_proba(self, X):
        dists = np.zeros((X.shape[0], len(self.classes_)))
        preds = self.predict(X)
        for i in range(0, X.shape[0]):
            dists[i, np.where(self.classes_ == preds[i])] = 1
        return dists

class StackerV3(BaseClassifier):
    def __init__(self, random_state=None, n_repetitions = 1, k_folds=10):
        super().__init__()
        self.n_repetitions = n_repetitions
        self.k_folds = k_folds
        self.random_state = random_state
        self.cv_splits = None

    def get_model(self, name):
        if name == 'multirocket-ridgecv':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'hydra_': SparseScaler(),
                        'multirocket_': StandardScaler()
                    },
                    verbose=False
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10))
            )
            return pipe
        elif name == 'rstsf':
            return RSTSF(random_state=self.random_state, n_jobs=-1, n_estimators=100)
        elif name == 'quant-etc':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'quant_': NoScaler(),
                    },
                    verbose=False
                ),
                ExtraTreesClassifier(
                    n_estimators=200,
                    max_features=0.1,
                    criterion="entropy",
                    random_state=self.random_state, # pass this in 
                    n_jobs=-1
                )
            )
            return pipe
        elif name == 'rdst-ridgecv':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'rdst_': StandardScaler(),
                    },
                    verbose=False
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-4, 4, 20))
            )
            return pipe
        elif name == 'rdst-robustscale-ridgecv':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'rdst_': RobustScaler(),
                    },
                    verbose=False
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-4, 4, 20))
            )
            return pipe
        elif name == 'catch22-quant-et':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'catch22_': NoScaler(),
                        'quant_': NoScaler(),
                    },
                    verbose=False
                ),
                ExtraTreesClassifier(
                    n_estimators=200,
                    max_features=0.1,
                    criterion="entropy",
                    random_state=self.random_state, # pass this in 
                    n_jobs=-1
                )
            )
            return pipe
        elif name == 'probability-linear-svc':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'proba_': StandardScaler(),
                    },
                    verbose=False
                ),
                SVC(kernel='linear', probability=True, random_state=self.random_state)
            )
            return pipe
        elif name == 'probability-ridgecv':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'proba_': StandardScaler(),
                    },
                    verbose=False
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10))
            )
            return pipe
        elif name == 'probability-rf':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'proba_': StandardScaler(),
                    },
                    verbose=False
                ),
                RandomForestClassifier(n_estimators=200, random_state=self.random_state, n_jobs=-1)
            )
            return pipe
        else:
            raise ValueError(f"Unknown model name: {name}")

    def fit_tranformers(self, X):
        self.t1 = RandomDilatedShapeletTransform(n_jobs=-1, random_state=self.random_state)
        self.t2 = QUANTTransformer()
        self.t3 = MultiRocket(n_jobs=-1, random_state=self.random_state)
        self.t4 = HydraTransformer(n_jobs=-1, random_state=self.random_state)
        self.t5 = Catch22(n_jobs=-1)
        self.t1.fit(X)
        self.t2.fit(X)
        self.t3.fit(X)
        self.t4.fit(X)
        self.t5.fit(X)

    def transform_series(self, X):
        
        start = perf_counter()
        Xt1 = self.t1.transform(X)
        t1_time = perf_counter() - start
        print(f"RDST transform: {t1_time:.4f}s")
        
        start = perf_counter()
        Xt2 = self.t2.transform(X)
        t2_time = perf_counter() - start
        print(f"QUANT transform: {t2_time:.4f}s")
        
        start = perf_counter()
        Xt3 = self.t3.transform(X)
        t3_time = perf_counter() - start
        print(f"MultiRocket transform: {t3_time:.4f}s")
        
        start = perf_counter()
        Xt4 = self.t4.transform(X)
        t4_time = perf_counter() - start
        print(f"Hydra transform: {t4_time:.4f}s")
        
        start = perf_counter()
        Xt5 = self.t5.transform(X)
        t5_time = perf_counter() - start
        print(f"Catch22 transform: {t5_time:.4f}s")

        return pl.DataFrame(np.hstack([Xt1, Xt2, Xt3, Xt4, Xt5]), schema=
            [f'rdst_{i}' for i in range(Xt1.shape[1])] + 
            [f'quant_{i}' for i in range(Xt2.shape[1])] + 
            [f'multirocket_{i}' for i in range(Xt3.shape[1])] + 
            [f'hydra_{i}' for i in range(Xt4.shape[1])] + 
            [f'catch22_{i}' for i in range(Xt5.shape[1])]
        )

    def _fit(self, X, y):
        if self.cv_splits is None:
            self.cv_splits = generate_folds(X, y, n_splits=self.k_folds, n_repetitions=self.n_repetitions, random_state=self.random_state)
        self.fit_tranformers(X)
        self.Xt_ = self.transform_series(X)
        self.trained_models_ = []

        self.tsc_algos = ['rstsf']
        self.feature_algos = ['multirocket-ridgecv', 'quant-etc', 'rdst-ridgecv', 'probability-ridgecv']

        for model_name in self.tsc_algos:
            oof_proba = []
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(X[train_idx], y[train_idx])
                proba = pipe.predict_proba(X[val_idx])

                prob_columns = []
                for idx, p in zip(val_idx, proba):
                    d = {
                        'index': idx,
                    }
                    for scls, prob in zip(pipe.classes_, p):
                        k = f'proba_model_{model_name}_class_{scls}'
                        d[k] = prob.item()
                        if k not in prob_columns:
                            prob_columns.append(k)
                    oof_proba.append(d)
                model_group.append(pipe)
            self.trained_models_.append((model_name, model_group))
            agg_probs = pl.DataFrame(oof_proba).group_by('index').mean().sort('index').select(prob_columns)
            self.Xt_ = pl.concat([self.Xt_, agg_probs], how='horizontal')

            # for each model compute oof accuracy
            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            print(f"Model {model_name}| CA: {oof_acc:.4f}")

        for model_name in self.feature_algos:
            oof_proba = []
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(self.Xt_[train_idx], y[train_idx])
                proba = pipe.predict_proba(self.Xt_[val_idx])

                prob_columns = []
                for idx, p in zip(val_idx, proba):
                    d = {
                        'index': idx,
                    }
                    for scls, prob in zip(pipe.classes_, p):
                        k = f'proba_model_{model_name}_class_{scls}'
                        d[k] = prob.item()
                        if k not in prob_columns:
                            prob_columns.append(k)
                    oof_proba.append(d)
                model_group.append(pipe)
            self.trained_models_.append((model_name, model_group))
            agg_probs = pl.DataFrame(oof_proba).group_by('index').mean().sort('index').select(prob_columns)
            self.Xt_ = pl.concat([self.Xt_, agg_probs], how='horizontal')

            # for each model compute oof accuracy
            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            # ocmute also log loss
            #from sklearn.metrics import log_loss, roc_auc_score
            #log_loss_value = log_loss(y, oof_probas)
            # ocmpute AUC 
            #if len(np.unique(y)) == 2:
            #    auc_value = roc_auc_score(y, oof_probas[:, 1])
            #else:
            #    auc_value = roc_auc_score(y, oof_probas, multi_class='ovr', average='macro')
            #print(f"Model {model_name}| CA: {oof_acc:.4f}, Log Loss: {log_loss_value:.4f}, AUC: {auc_value:.4f}")
            print(f"Model {model_name}| CA: {oof_acc:.4f}")

    def _predict_proba(self, X):
        Xt = self.transform_series(X)

        for model_name, model_group in self.trained_models_:
            oof_proba = []
            for model in model_group:
                if model_name in self.tsc_algos:
                    proba = model.predict_proba(X)
                else:
                    proba = model.predict_proba(Xt)
                prob_columns = []
                for i, p in enumerate(proba):
                    d = {
                        'index': i,
                    }
                    for scls, prob in zip(model.classes_, p):
                        k = f'proba_model_{model_name}_class_{scls}'
                        d[k] = prob.item()
                        if k not in prob_columns:
                            prob_columns.append(k)
                    oof_proba.append(d)
            df = pl.DataFrame(oof_proba).group_by('index').mean().sort('index').select(prob_columns)
            Xt = pl.concat([Xt, df], how='horizontal')

        # return probabilities form last model TODO make this best model
        return Xt.select(prob_columns).to_numpy()

    def _predict(self, X):
        probas = self._predict_proba(X)
        predicted_indices = np.argmax(probas, axis=1)
        return self.classes_[predicted_indices]

In [4]:
random_state = 149
n_repetitions=1
s1 = Stacker(random_state=random_state, n_repetitions=n_repetitions)
s2 = StackerV3(random_state=random_state, n_repetitions=n_repetitions)

In [5]:
s2.fit(X_train, y_train)
pred = s2.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

RDST transform: 4.5284s
QUANT transform: 5.0049s
MultiRocket transform: 2.2042s
Hydra transform: 2.7209s
Catch22 transform: 1.1235s


100%|██████████| 10/10 [00:58<00:00,  5.87s/it]


Model rstsf| CA: 0.7348


100%|██████████| 10/10 [00:26<00:00,  2.61s/it]


Model multirocket-ridgecv| CA: 0.7514


...

100%|██████████| 10/10 [00:08<00:00,  1.11it/s]


Model rdst-ridgecv| CA: 0.7348


100%|██████████| 10/10 [00:01<00:00,  6.99it/s]


Model probability-ridgecv| CA: 0.7569
RDST transform: 2.7022s
QUANT transform: 1.7722s
MultiRocket transform: 0.2183s
Hydra transform: 1.2754s
Catch22 transform: 0.1659s


TypeError: ERROR passed input of type <class 'polars.dataframe.frame.DataFrame'>, must be of type np.ndarray, pd.DataFrame or list of np.ndarray/pd.DataFrame.See aeon.utils.data_types.COLLECTIONS_DATA_TYPES

In [ ]:
s2.Xt_

In [ ]:
m = MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.interval_based import RSTSF
m = RSTSF(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)    
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.hybrid import HIVECOTEV2
m = HIVECOTEV2(time_limit_in_minutes=10, random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.dictionary_based import TemporalDictionaryEnsemble
m = TemporalDictionaryEnsemble(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)    
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.shapelet_based import ShapeletTransformClassifier
m = ShapeletTransformClassifier(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)    
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.interval_based import DrCIFClassifier
m = DrCIFClassifier(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)    
acc = accuracy_score(y_test, pred)
acc

In [ ]:
X_train.shape

In [ ]:
from aeon.transformations.collection.shapelet_based import SAST
from aeon.datasets import load_unit_test

sast = SAST(n_jobs=-1)
sast.fit(X_train, y_train)

sast.transform(X_train)

In [ ]:
fsdf=dfgdgf

In [ ]:
s1.fit(X_train, y_train)
pred = s1.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
import numpy as np
from sklearn.linear_model import RidgeClassifierCV



# Generate random data
np.random.seed(42)
X_train = np.random.randn(100, 20)  # 100 samples, 20 features
y_train = np.random.randint(0, 3, 100)  # 3 classes

X_test = np.random.randn(30, 20)
y_test = np.random.randint(0, 3, 30)

# Train RidgeClassifierCV
model = RidgeClassifierCVIndicator(alphas=[0.1, 1.0, 10.0, 100.0])
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)
score = model.score(X_test, y_test)

print(f"Best alpha: {model.alpha_}")
print(f"Test accuracy: {score:.3f}")
print(f"Predictions: {predictions[:10]}")

In [ ]:
model.predict_proba(X_test)

In [ ]:
model.predict(X_test)